<a href="https://colab.research.google.com/github/yi-juho/AI_LLM_BERT/blob/main/KAN_colab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn.functional as F
import math


class KANLinear(torch.nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        enable_standalone_scale_spline=True,
        #base_activation=CustomSiLUApprox123,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KANLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (
                torch.arange(-spline_order, grid_size + spline_order + 1) * h
                + grid_range[0]
            )
            .expand(in_features, -1)
            .contiguous()
        )
        self.register_buffer("grid", grid)

        self.base_weight = torch.nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = torch.nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            self.spline_scaler = torch.nn.Parameter(
                torch.Tensor(out_features, in_features)
            )

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = (
                (
                    torch.rand(self.grid_size + 1, self.in_features, self.out_features)
                    - 1 / 2
                )
                * self.scale_noise
                / self.grid_size
            )
            self.spline_weight.data.copy_(
                (self.scale_spline if not self.enable_standalone_scale_spline else 1.0)
                * self.curve2coeff(
                    self.grid.T[self.spline_order : -self.spline_order],
                    noise,
                )
            )
            if self.enable_standalone_scale_spline:
                # torch.nn.init.constant_(self.spline_scaler, self.scale_spline)
                torch.nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5) * self.scale_spline)

    def b_splines(self, x: torch.Tensor):
        """
        Compute the B-spline bases for the given input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).

        Returns:
            torch.Tensor: B-spline bases tensor of shape (batch_size, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features

        grid: torch.Tensor = (
            self.grid
        )  # (in_features, grid_size + 2 * spline_order + 1)
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = (
                (x - grid[:, : -(k + 1)])
                / (grid[:, k:-1] - grid[:, : -(k + 1)])
                * bases[:, :, :-1]
            ) + (
                (grid[:, k + 1 :] - x)
                / (grid[:, k + 1 :] - grid[:, 1:(-k)])
                * bases[:, :, 1:]
            )

        assert bases.size() == (
            x.size(0),
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return bases.contiguous()

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):
        """
        Compute the coefficients of the curve that interpolates the given points.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).
            y (torch.Tensor): Output tensor of shape (batch_size, in_features, out_features).

        Returns:
            torch.Tensor: Coefficients tensor of shape (out_features, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features
        assert y.size() == (x.size(0), self.in_features, self.out_features)

        A = self.b_splines(x).transpose(
            0, 1
        )  # (in_features, batch_size, grid_size + spline_order)
        B = y.transpose(0, 1)  # (in_features, batch_size, out_features)
        solution = torch.linalg.lstsq(
            A, B
        ).solution  # (in_features, grid_size + spline_order, out_features)
        result = solution.permute(
            2, 0, 1
        )  # (out_features, in_features, grid_size + spline_order)

        assert result.size() == (
            self.out_features,
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return result.contiguous()

    @property
    def scaled_spline_weight(self):
        return self.spline_weight * (
            self.spline_scaler.unsqueeze(-1)
            if self.enable_standalone_scale_spline
            else 1.0
        )

    def forward(self, x: torch.Tensor):
        assert x.size(-1) == self.in_features
        original_shape = x.shape
        x = x.reshape(-1, self.in_features)

        base_output = F.linear(self.base_activation(x), self.base_weight)
        spline_output = F.linear(
            self.b_splines(x).view(x.size(0), -1),
            self.scaled_spline_weight.view(self.out_features, -1),
        )
        output = base_output + spline_output

        output = output.reshape(*original_shape[:-1], self.out_features)
        return output

    @torch.no_grad()
    def update_grid(self, x: torch.Tensor, margin=0.01):
        assert x.dim() == 2 and x.size(1) == self.in_features
        batch = x.size(0)

        splines = self.b_splines(x)  # (batch, in, coeff)
        splines = splines.permute(1, 0, 2)  # (in, batch, coeff)
        orig_coeff = self.scaled_spline_weight  # (out, in, coeff)
        orig_coeff = orig_coeff.permute(1, 2, 0)  # (in, coeff, out)
        unreduced_spline_output = torch.bmm(splines, orig_coeff)  # (in, batch, out)
        unreduced_spline_output = unreduced_spline_output.permute(
            1, 0, 2
        )  # (batch, in, out)

        # sort each channel individually to collect data distribution
        x_sorted = torch.sort(x, dim=0)[0]
        grid_adaptive = x_sorted[
            torch.linspace(
                0, batch - 1, self.grid_size + 1, dtype=torch.int64, device=x.device
            )
        ]

        uniform_step = (x_sorted[-1] - x_sorted[0] + 2 * margin) / self.grid_size
        grid_uniform = (
            torch.arange(
                self.grid_size + 1, dtype=torch.float32, device=x.device
            ).unsqueeze(1)
            * uniform_step
            + x_sorted[0]
            - margin
        )

        grid = self.grid_eps * grid_uniform + (1 - self.grid_eps) * grid_adaptive
        grid = torch.concatenate(
            [
                grid[:1]
                - uniform_step
                * torch.arange(self.spline_order, 0, -1, device=x.device).unsqueeze(1),
                grid,
                grid[-1:]
                + uniform_step
                * torch.arange(1, self.spline_order + 1, device=x.device).unsqueeze(1),
            ],
            dim=0,
        )

        self.grid.copy_(grid.T)
        self.spline_weight.data.copy_(self.curve2coeff(x, unreduced_spline_output))

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        """
        Compute the regularization loss.

        This is a dumb simulation of the original L1 regularization as stated in the
        paper, since the original one requires computing absolutes and entropy from the
        expanded (batch, in_features, out_features) intermediate tensor, which is hidden
        behind the F.linear function if we want an memory efficient implementation.

        The L1 regularization is now computed as mean absolute value of the spline
        weights. The authors implementation also includes this term in addition to the
        sample-based regularization.
        """
        l1_fake = self.spline_weight.abs().mean(-1)
        regularization_loss_activation = l1_fake.sum()
        p = l1_fake / regularization_loss_activation
        regularization_loss_entropy = -torch.sum(p * p.log())
        return (
            regularize_activation * regularization_loss_activation
            + regularize_entropy * regularization_loss_entropy
        )


class KAN(torch.nn.Module):
    def __init__(
        self,
        layers_hidden,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        #base_activation=CustomSiLUApprox123,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KAN, self).__init__()
        self.grid_size = grid_size
        self.spline_order = spline_order

        self.layers = torch.nn.ModuleList()
        for in_features, out_features in zip(layers_hidden, layers_hidden[1:]):
            self.layers.append(
                KANLinear(
                    in_features,
                    out_features,
                    grid_size=grid_size,
                    spline_order=spline_order,
                    scale_noise=scale_noise,
                    scale_base=scale_base,
                    scale_spline=scale_spline,
                    base_activation=base_activation,
                    grid_eps=grid_eps,
                    grid_range=grid_range,
                )
            )

    def forward(self, x: torch.Tensor, update_grid=False):
        for layer in self.layers:
            if update_grid:
                layer.update_grid(x)
            x = layer(x)
        return x

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        return sum(
            layer.regularization_loss(regularize_activation, regularize_entropy)
            for layer in self.layers
        )

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
#from tqdm import tqdm
import matplotlib.pyplot as plt

# 데이터 생성
def generate_data(func, n_samples=1000):
    x = torch.linspace(-1, 1, n_samples).view(-1,1)
    y = func(x)
    return x, y


def target_function(x_tensor):
    # [-1, 1] 구간에서 경계(±1)에서 0이 되도록 Hann window 적용 후,
    # 내부에 3사이클 사인파를 넣어 적당한 난이도 유지
    x = x_tensor.view(-1, 1)
    envelope = 0.5 * (1.0 + torch.cos(math.pi * x))   # 0 at x=±1, 1 at x=0
    y = envelope * torch.sin(3.0 * math.pi * x)       # 3 cycles on [-1,1]
    return y

# 데이터 준비
x_train, y_train = generate_data(target_function, n_samples=800)
x_val, y_val = generate_data(target_function, n_samples=200)

# 모델 정의
model = KAN([1,16,1])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)



# 옵티마이저 및 손실 함수
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# 학습 루프
for epoch in range(5000):
    model.train()
    optimizer.zero_grad()
    pred = model(x_train.to(device))
    loss = criterion(pred, y_train.to(device))
    loss.backward()
    optimizer.step()

    # 검증
    model.eval()
    with torch.no_grad():
        val_pred = model(x_val.to(device))
        val_loss = criterion(val_pred, y_val.to(device)).item()

    if (epoch+1) % 1000 == 0:
        print(f"[Epoch {epoch+1}] Train Loss: {loss.item():.4f} | Val Loss: {val_loss:.4f}")

# 결과 시각화
model.eval()
with torch.no_grad():
    x_test = torch.linspace(-1, 1, 1000).view(-1, 1).to(device)
    y_pred = model(x_test).cpu()

plt.plot(x_test.cpu(), target_function(x_test.cpu()), label="True Function")
plt.plot(x_test.cpu(), y_pred, label="KAN Approximation")
plt.legend()
plt.title("Function Approximation by KAN")
plt.grid(True)
plt.show()



In [ ]:
import os
torch.save(model.state_dict(), "func.pth")
print("✅ 모델이 model/efficient_kan_svhn.pth 경로에 저장되었습니다.")

In [ ]:
# 모델 초기화 (동일한 구조로)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KAN([1,16,1])
model.load_state_dict(torch.load("func.pth", map_location=device))
model.eval()


In [ ]:
import torch
import torch.nn as nn
import math


# --- 데이터 생성 함수 (학습 때와 동일하게)
def generate_data(func, n_samples=1000):
    x = torch.linspace(-1, 1, n_samples).view(-1, 1)
    y = func(x)
    return x, y

def target_function(x_tensor):
    x_list = x_tensor.view(-1).tolist()
    y_list = []
    for x in x_list:
        envelope = (1 + math.cos(2 * math.pi * x)) / 2
        val = envelope * math.sin(20 * math.pi * x)
        y_list.append(val)
    return torch.tensor(y_list, dtype=torch.float32).view(-1, 1)

# --- 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 모델 불러오기
model = KAN([1,16,1]).to(device)
model.load_state_dict(torch.load("func.pth", map_location=device))
model.eval()

# --- Validation 데이터 준비
x_val, y_val = generate_data(target_function, n_samples=200)
x_val, y_val = x_val.to(device), y_val.to(device)

# --- Validation 수행
criterion = nn.MSELoss()
with torch.no_grad():
    val_pred = model(x_val)
    val_loss = criterion(val_pred, y_val).item()

print(f"불러온 모델의 Validation Loss: {val_loss:.4f}")


In [ ]:
import torch

# ===== Q5.11 고정소수 변환/출력 유틸 =====
FRAC_BITS = 11
SCALE = 1 << FRAC_BITS  # 2048

def round_half_up_q(v: torch.Tensor):
    """half-up 반올림: ties -> +inf 방향 (bankers 아님)"""
    # v: float
    return torch.where(v >= 0, torch.floor(v + 0.5), torch.ceil(v - 0.5))

def floats_to_q5_11_int16(t: torch.Tensor):
    """float -> Q5.11 정수(int16 범위로 clamp)"""
    q = round_half_up_q(t * SCALE).to(torch.int32)
    q = torch.clamp(q, -32768, 32767)
    return q

def ints_to_hex16(q: torch.Tensor):
    """int -> 16비트 2의보수 HEX(4자리) 리스트"""
    u = (q.to(torch.int32) & 0xFFFF).reshape(-1).cpu().tolist()
    return [f"{x:04X}" for x in u]

def floats_to_hex16_q5_11(t: torch.Tensor):
    """float -> Q5.11 양자화 후 HEX(4자리) 리스트"""
    return ints_to_hex16(floats_to_q5_11_int16(t))

def print_hex16(name: str, t: torch.Tensor, n: int = 16):
    flat = t.reshape(-1)[:n]
    if flat.dtype.is_floating_point:
        hexs = floats_to_hex16_q5_11(flat)
    else:
        hexs = ints_to_hex16(flat)
    print(f"{name}: {' '.join(hexs)}")

# 16비트 이진(Q5.11) → float
def bin16_to_int(s: str) -> int:
    v = int(s, 2)
    return v - 0x10000 if (v & 0x8000) else v

def bin16_to_q5_11_float(s: str) -> float:
    return bin16_to_int(s) / float(SCALE)

# ===== 모델 중간 출력 캡처 =====
def get_module_by_path(root, path: str):
    """'layers.0' 같은 경로로 서브모듈 얻기"""
    m = root
    for p in path.split("."):
        m = m[int(p)] if p.isdigit() else getattr(m, p)
    return m

# ===== 설정 =====
LAYER_PATH = "layers.0"         # <- 중간값을 보고 싶은 모듈 경로 (필요시 수정)
BIN_X      = "0001000000000001" # 입력 비트열

# ===== 입력 생성: (1,16,1) =====
x_val = bin16_to_q5_11_float(BIN_X)   # ≈ 4097/2048 ≈ 2.000488...
x     = torch.full((1,16,1), x_val, dtype=torch.float32)

# ===== 실행 & 중간 출력 16개 HEX로 보기 =====
model.eval()
buf = {}

# 입력이 제대로 들어가는지(선택) 첫 레이어 "입력"도 캡처
h_in  = get_module_by_path(model, LAYER_PATH).register_forward_pre_hook(
    lambda m, inp: buf.setdefault("layer_in", inp[0].detach())
)
# 우리가 원하는 "중간 출력" 캡처
h_out = get_module_by_path(model, LAYER_PATH).register_forward_hook(
    lambda m, inp, out: buf.setdefault("layer_out", out.detach())
)

with torch.no_grad():
    _ = model(x)

h_in.remove(); h_out.remove()

# 출력
print_hex16("x_broadcast(Q5.11)", x, 16)              # 입력이 기대(1001)인지 확인용
print_hex16(f"{LAYER_PATH}__in",  buf["layer_in"], 16)
print_hex16(f"{LAYER_PATH}__out", buf["layer_out"], 16)  # ← 여기 16개가 '중간 결과값'


In [ ]:
import torch
import torch.nn.functional as F

# === 모델 로드 ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
full = KAN([1,16,1]).to(device)
full.load_state_dict(torch.load("func.pth", map_location=device))
full.eval()

# === 관찰 설정 ===
layer_idx = 0     # 첫 레이어
out_idx   = 0     # 관찰할 노드(0~15)
in_pick   = 0     # in_features>1일 때 보여줄 입력 피처 인덱스
x_in = torch.tensor([[0.5]], dtype=torch.float32, device=device)  # (B=1, in=1 예시)

layer = full.layers[layer_idx]

with torch.no_grad():
    # (1) 내부 중간값
    silu  = layer.base_activation(x_in)      # (B, in)
    bases = layer.b_splines(x_in)            # (B, in, K)  ← 최종 3차 기저값

    # (2) 가중치(계수) 추출
    #  - W_silu: SiLU에 곱해지는 가중치(입력 피처별)
    #  - C_k: 각 입력 피처의 각 기저에 대한 "계수"
    W_silu = layer.base_weight[out_idx, :]               # (in,)
    C_k    = layer.scaled_spline_weight[out_idx, :, :]   # (in, K)

    # (3) 브랜치별 출력 (노드 out_idx의 스칼라 출력)
    #  - SiLU 가지: sum_j SiLU(x_j) * W_silu[j]
    base_out = (silu * W_silu.unsqueeze(0)).sum(dim=1, keepdim=True)          # (B,1)

    #  - Spline 가지: sum_{j,k} W_spline[j,k](x) * C_k[j,k]
    spline_out = (bases * C_k.unsqueeze(0)).sum(dim=(1,2), keepdim=True)      # (B,1)

    y_out = base_out + spline_out                                             # (B,1)

# ===== 출력 =====
B = x_in.size(0)
K = bases.size(-1)

print("=== Inputs (to the node, your naming) ===")
print(f"x_in (batch {B}, in_features {layer.in_features}): {x_in.tolist()}")

print("\nW_silu ----------------------------------")
print(f"  shape: {tuple(W_silu.shape)}")
print(f"  values: {W_silu.tolist()}")

print("\nC_k -------------------------------------")
print(f"  shape (K={K}): {tuple(C_k[in_pick].shape)}")
for k, coef in enumerate(C_k[in_pick].tolist()):
    print(f"  C_{k}: {coef:.7f}")

print("\n== W_spline (final spline basis values per Ck, chosen input feature) --")
print(f"  take in_feature index = {in_pick}")
print(f"  shape (K={K}): {tuple(bases[0, in_pick].shape)}")
for k, val in enumerate(bases[0, in_pick, :].tolist()):
    print(f"  W_spline[{k}]: {val:.7f}")

print("\n=== Outputs (from the node) ===")
print(f"silu branch output (Σ_j SiLU(x_j)·W_silu_j): {base_out.item():.7f}")
print(f"spline branch output (Σ_{'{j,k}'} W_spline_{'{j,k}'}(x)·C_{'{j,k}'}): {spline_out.item():.7f}")
print(f"final output (sum)                      : {y_out.item():.7f}")


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = KAN([1,16,1]).to(device)
model.load_state_dict(torch.load("func.pth", map_location=device))
model.eval()

x_input = torch.tensor([[0.5]], dtype=torch.float32, device=device)  # (1,1)

with torch.no_grad():
    h = model.layers[0](x_input)  # (1,4) ← 여기가 "각 4개 노드 출력"
    y = model.layers[1](h)        # (1,1) 최종 출력

print("hidden(4):", h.squeeze(0).tolist())

print("y:", y.item())


In [ ]:
import math
import csv
from typing import Iterable, Union, List, Dict

Number = Union[float, int]

# ===== 반올림 모드 설정 =====
# 'half_up'  : 0.5 절단점은 0에서 멀어지게 (이전 PyTorch 유틸과 동일)
# 'bankers'  : 파이썬 round() (half-to-even)
ROUND_MODE = 'half_up'

def round_half_up(x: float) -> int:
    """최근접 반올림, 정확히 .5는 0으로부터 멀어지게"""
    return math.floor(x + 0.5) if x >= 0 else math.ceil(x - 0.5)

def q_round(x: float) -> int:
    if ROUND_MODE == 'half_up':
        return round_half_up(x)
    elif ROUND_MODE == 'bankers':
        return int(round(x))  # Python half-to-even
    else:
        raise ValueError("ROUND_MODE must be 'half_up' or 'bankers'.")

def float_to_qmn(
    val: Number,
    word_bits: int = 16,
    frac_bits: int = 11,
    saturate: bool = True
) -> Dict[str, Union[str, int, float]]:
    """
    float -> Qm.n (기본: Q5.11, 16비트) 변환
    - 반올림: ROUND_MODE로 제어 (기본 half_up = 이전 PyTorch 유틸과 동일)
    - 반환: 정수값(q_int), 2의보수 이진(bin), HEX(hex), 복원값(dequant)
    """
    assert word_bits >= 2 and 0 <= frac_bits < word_bits, "비트 설정이 올바르지 않습니다."
    scale = 1 << frac_bits
    min_int = -(1 << (word_bits - 1))
    max_int =  (1 << (word_bits - 1)) - 1

    # half-up(or bankers) 라운드 후 정수화
    q_int = q_round(float(val) * scale)

    # 포화(saturation) (표준 Q 포맷 클램프)
    if saturate:
        q_int = max(min(q_int, max_int), min_int)
    else:
        # wrap-around가 필요하면 여기서 처리 (일반적으로 권장 X)
        mask = (1 << word_bits) - 1
        q_int = ((q_int + (1 << word_bits)) & mask) - (1 << (word_bits - 1))

    # 2의 보수 unsigned 표현으로 BIN/HEX 문자열 생성
    unsigned = q_int & ((1 << word_bits) - 1)
    bin_str = format(unsigned, f'0{word_bits}b')
    hex_str = format(unsigned, f'0{(word_bits+3)//4}X')

    # 복원값(양자화 오차 확인용)
    dequant = q_int / scale

    return {
        "q_int": q_int,
        "bin": bin_str,
        "hex": hex_str,
        "dequant": dequant
    }

def qmn_convert_table(
    names: Iterable[str],
    values: Iterable[Number],
    word_bits: int = 16,
    frac_bits: int = 11
) -> List[Dict[str, Union[str, int, float]]]:
    """여러 값 일괄 변환해 표(리스트[dict])로 반환"""
    table = []
    for n, v in zip(names, values):
        conv = float_to_qmn(v, word_bits=word_bits, frac_bits=frac_bits)
        table.append({
            "name": n,
            "float": float(v),
            "q_int": conv["q_int"],
            "bin": conv["bin"],
            "hex": conv["hex"],
            "dequant": conv["dequant"],
        })
    return table

def save_table_csv(table: List[Dict[str, Union[str, int, float]]], path: str) -> None:
    """표를 CSV로 저장 (pandas 없이 표준 csv 모듈 사용)"""
    if not table:
        return
    header = list(table[0].keys())
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=header)
        writer.writeheader()
        writer.writerows(table)
    print(f"Saved CSV -> {path}")

# ==========================
# 예시: 질문에서 준 값들
# ==========================
if __name__ == "__main__":
    x = -0.4989
    hidden = [
        0.42035651206970215, -0.1674988567829132, -0.09244704246520996, -0.3374970555305481,
        0.153248593211174, 0.31988227367401123, 0.30715101957321167, 0.406909704208374,
        -0.37123072147369385, 0.14589305222034454, 0.2754025161266327, -0.2121015191078186,
        -0.28645074367523193, -0.29311802983283997, -0.44083017110824585, 0.21694418787956238
    ]
    y = 0.49994078278541565

    names = ["x_input"] + [f"h{i}" for i in range(1, 17)] + ["y"]
    values = [x] + hidden + [y]

    table = qmn_convert_table(names, values, word_bits=16, frac_bits=11)

    # 콘솔 출력(간단 확인)
    for row in table:
        print(f"{row['name']:>8} | float={row['float']:+.9f} | q_int={row['q_int']:>6} | bin={row['bin']} | hex={row['hex']} | deq={row['dequant']:+.9f}")



In [ ]:
import torch
import torch.nn as nn
import math

# --- 데이터 함수 그대로 유지 ---
def target_function(x_tensor):
    # [-1, 1] 구간에서 경계(±1)에서 0이 되도록 Hann window 적용 후,
    # 내부에 3사이클 사인파를 넣어 적당한 난이도 유지
    x = x_tensor.view(-1, 1)
    envelope = 0.5 * (1.0 + torch.cos(math.pi * x))   # 0 at x=±1, 1 at x=0
    y = envelope * torch.sin(3.0 * math.pi * x)       # 3 cycles on [-1,1]
    return y

# --- 디바이스 설정 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 모델 불러오기 ---
model = KAN([1,16, 1]).to(device)
model.load_state_dict(torch.load("func.pth", map_location=device))
model.eval()

# --- x 하나 넣어서 예측 ---
x_input = torch.tensor([[0.26537]], dtype=torch.float32).to(device)  # 예: x=0.25
with torch.no_grad():
    y_pred = model(x_input).item()

print(f"x={x_input.item():.4f} → 예측 y={y_pred:.6f}")

# (옵션) 실제 타깃 값 비교
y_true = target_function(x_input.cpu()).item()
print(f"실제 y={y_true:.6f}")


In [ ]:
import torch
import numpy as np

# 1) state_dict 로드
file_path = "func.pth"
state_dict = torch.load(file_path, map_location="cpu")

# 2) layer0 파라미터 추출
base   = state_dict["layers.0.base_weight"].cpu().numpy()      # (O, I)
spline = state_dict["layers.0.spline_weight"].cpu().numpy()    # (O, I, K)
scaler = state_dict["layers.0.spline_scaler"].cpu().numpy()    # (O, I)

# 3) shape 확인
print("base shape:", base.shape)
print("spline shape:", spline.shape)
print("scaler shape:", scaler.shape)

# 4) 2D로 펼치기 (하드코딩X)
K = spline.shape[-1]
base_flat   = base.reshape(-1, 1)          # (O*I, 1)
scaler_flat = scaler.reshape(-1, 1)        # (O*I, 1)
spline_flat = spline.reshape(-1, K)        # (O*I, K)

# 5) 저장 순서: base, scaler, spline
combined = np.concatenate([base_flat, scaler_flat, spline_flat], axis=1)  # (O*I, 1+1+K)

# 6) CSV 저장
np.savetxt("layer0_base_scaler_spline.csv", combined, delimiter=",")

print("✅ 저장 완료: layer0_base_scaler_spline.csv")


In [ ]:
import torch
import numpy as np

# 1) state_dict 로드
file_path = "func.pth"
state_dict = torch.load(file_path, map_location="cpu")

def export_layer(state_dict, layer_idx: int):
    """
    layers.{idx}.base_weight   : (O, I)
    layers.{idx}.spline_weight : (O, I, K)
    layers.{idx}.spline_scaler : (O, I)
    => CSV로 (O*I, 1 + 1 + K) = (O*I, K+2) 형태로 저장
       저장 순서: base, scaler, spline
    """
    base_key   = f"layers.{layer_idx}.base_weight"
    spline_key = f"layers.{layer_idx}.spline_weight"
    scaler_key = f"layers.{layer_idx}.spline_scaler"

    if base_key not in state_dict:
        raise KeyError(f"키 없음: {base_key}")
    if spline_key not in state_dict:
        raise KeyError(f"키 없음: {spline_key}")
    if scaler_key not in state_dict:
        raise KeyError(f"키 없음: {scaler_key}")

    base   = state_dict[base_key].cpu().numpy()        # (O, I)
    spline = state_dict[spline_key].cpu().numpy()      # (O, I, K)
    scaler = state_dict[scaler_key].cpu().numpy()      # (O, I)

    O, I = base.shape
    _, _, K = spline.shape

    print(f"[layer {layer_idx}] base: {base.shape}, spline: {spline.shape}, scaler: {scaler.shape}")

    # 2D로 펼치기
    base_flat   = base.reshape(-1, 1)      # (O*I, 1)
    scaler_flat = scaler.reshape(-1, 1)    # (O*I, 1)
    spline_flat = spline.reshape(-1, K)    # (O*I, K)

    # 저장 순서: base, scaler, spline
    combined = np.concatenate([base_flat, scaler_flat, spline_flat], axis=1)  # (O*I, K+2)

    out_csv = f"layer{layer_idx}_base_scaler_spline{K}.csv"
    np.savetxt(out_csv, combined, delimiter=",")
    print(f"✅ 저장 완료: {out_csv}")

# 2) layer 0, layer 1 내보내기
for idx in [0, 1]:
    export_layer(state_dict, idx)


# **32bit csv파일로 변환**

In [ ]:
import csv
from decimal import Decimal, ROUND_HALF_UP, getcontext

# 정밀도 충분히 키움
getcontext().prec = 50

SCALE = 1 << 20  # 2**20
# 11.20 형식의 표현 가능 범위 (부호 포함)
MIN_VAL = Decimal(-2048)           # -2^11
MAX_VAL = Decimal(2047) + (Decimal(SCALE - 1) / Decimal(SCALE))  # 2047 + (1 - 2^-20)

def str_to_bits_11_20(s: str) -> str:
    """문자열 실수를 11.20 고정소수점 32비트(2의보수) 이진 문자열로."""
    s = s.strip()
    if not s:
        return None
    # 1) 문자열 그대로 Decimal로 파싱 (float() 사용 금지)
    v = Decimal(s)

    # 2) 표현 범위 포화
    if v < MIN_VAL: v = MIN_VAL
    if v > MAX_VAL: v = MAX_VAL

    # 3) 스케일 적용 후 half-up 반올림 (은행가 반올림 방지)
    scaled = (v * Decimal(SCALE)).to_integral_value(rounding=ROUND_HALF_UP)

    # 4) 32비트 2의 보수로 변환
    twos = int(scaled) & 0xFFFFFFFF
    return format(twos, '032b')

# ── 단일 값 검증 ──
manual = "-6.65E-01"  # 직접 넣은 값(입력 문자열 그대로)
print("manual bits:", str_to_bits_11_20(manual))

# ── CSV → 텍스트(엑셀이 텍스트로 인식하도록 앞에 ' 붙이기) ──
inp = "layer0_base_scaler_spline8.csv"
out = "layer0_base_scaler_spline8_bits.csv"
with open(inp, "r", newline="") as fin, open(out, "w", newline="") as fout:
    r = csv.reader(fin)
    w = csv.writer(fout)
    for row in r:
        if not row:
            continue
        bits = [("'" + str_to_bits_11_20(x)) for x in row if x.strip()]
        w.writerow(bits)

print("✅ 저장 완료:", out)


In [ ]:
import csv
from decimal import Decimal, ROUND_HALF_UP, getcontext

# 정밀도 충분히 키움
getcontext().prec = 50

SCALE = 1 << 20  # 2**20
# 11.20 형식의 표현 가능 범위 (부호 포함)
MIN_VAL = Decimal(-2048)           # -2^11
MAX_VAL = Decimal(2047) + (Decimal(SCALE - 1) / Decimal(SCALE))  # 2047 + (1 - 2^-20)

def str_to_bits_11_20(s: str) -> str:
    """문자열 실수를 11.20 고정소수점 32비트(2의보수) 이진 문자열로."""
    s = s.strip()
    if not s:
        return None
    # 1) 문자열 그대로 Decimal로 파싱 (float() 사용 금지)
    v = Decimal(s)

    # 2) 표현 범위 포화
    if v < MIN_VAL: v = MIN_VAL
    if v > MAX_VAL: v = MAX_VAL

    # 3) 스케일 적용 후 half-up 반올림 (은행가 반올림 방지)
    scaled = (v * Decimal(SCALE)).to_integral_value(rounding=ROUND_HALF_UP)

    # 4) 32비트 2의 보수로 변환
    twos = int(scaled) & 0xFFFFFFFF
    return format(twos, '032b')

# ── 단일 값 검증 ──
manual = "-6.65E-01"  # 직접 넣은 값(입력 문자열 그대로)
print("manual bits:", str_to_bits_11_20(manual))

# ── CSV → 텍스트(엑셀이 텍스트로 인식하도록 앞에 ' 붙이기) ──
inp = "layer1_base_scaler_spline8.csv"
out = "layer1_base_scaler_spline8_bits.csv"
with open(inp, "r", newline="") as fin, open(out, "w", newline="") as fout:
    r = csv.reader(fin)
    w = csv.writer(fout)
    for row in r:
        if not row:
            continue
        bits = [("'" + str_to_bits_11_20(x)) for x in row if x.strip()]
        w.writerow(bits)

print("✅ 저장 완료:", out)


In [ ]:
import csv
from decimal import Decimal, ROUND_HALF_UP, getcontext

# 충분한 정밀도
getcontext().prec = 50

# Q5.11 설정
FRAC_BITS = 11
SCALE = 1 << FRAC_BITS           # 2^11 = 2048
MIN_VAL = Decimal(-16)           # -2^(16-11-1) = -16
MAX_VAL = Decimal(15) + (Decimal(SCALE - 1) / Decimal(SCALE))  # 15 + (2047/2048)

def str_to_bits_q5_11(s: str) -> str:
    """문자열 실수 → Q5.11(16비트, 2의 보수) 이진 문자열 16자리."""
    s = s.strip()
    if not s:
        return None
    v = Decimal(s)

    # 포화
    if v < MIN_VAL: v = MIN_VAL
    if v > MAX_VAL: v = MAX_VAL

    # 스케일 & Half-up 반올림
    scaled = (v * Decimal(SCALE)).to_integral_value(rounding=ROUND_HALF_UP)

    # 16비트 2의 보수
    twos = int(scaled) & 0xFFFF
    return format(twos, '016b')

# ── 단일 값 검증 ──
manual = "-6.65E-01"
print("manual bits (Q5.11):", str_to_bits_q5_11(manual))

# ── CSV 변환 (엑셀에서 텍스트 인식 위해 앞에 ' 붙임) ──
inp = "layer0_base_scaler_spline8.csv"
out = "layer0_base_scaler_spline8_bits16.csv"
with open(inp, "r", newline="") as fin, open(out, "w", newline="") as fout:
    r = csv.reader(fin)
    w = csv.writer(fout)
    for row in r:
        if not row:
            continue
        bits = [("'" + str_to_bits_q5_11(x)) for x in row if x.strip()]
        w.writerow(bits)

print("✅ 저장 완료:", out)


In [ ]:
import csv
from decimal import Decimal, ROUND_HALF_UP, getcontext

# 충분한 정밀도
getcontext().prec = 50

# Q5.11 설정
FRAC_BITS = 11
SCALE = 1 << FRAC_BITS                      # 2^11 = 2048
MIN_VAL = Decimal(-16)                      # -2^(5-1) = -16
MAX_VAL = Decimal(15) + Decimal(SCALE - 1) / Decimal(SCALE)  # 15 + 2047/2048

def str_to_bits_q5_11(s: str) -> str:
    """문자열 실수 → Q5.11(16비트, 2의 보수) 이진 문자열 16자리."""
    s = s.strip()
    if not s:
        return None

    # 문자열을 Decimal로 안전하게 파싱
    v = Decimal(s)

    # 포화 (saturation)
    if v < MIN_VAL:
        v = MIN_VAL
    if v > MAX_VAL:
        v = MAX_VAL

    # 스케일 및 half-up 반올림
    scaled = (v * Decimal(SCALE)).to_integral_value(rounding=ROUND_HALF_UP)

    # 16비트 2의 보수로 변환
    twos = int(scaled) & 0xFFFF
    return format(twos, '016b')

# ── 단일 값 검증 ──
manual = "-6.65E-01"
print("manual bits (Q5.11):", str_to_bits_q5_11(manual))

# ── CSV → 텍스트(엑셀이 텍스트로 인식하도록 앞에 ' 붙이기) ──
inp = "layer1_base_scaler_spline8.csv"
out = "layer1_base_scaler_spline8_bits16.csv"
with open(inp, "r", newline="") as fin, open(out, "w", newline="") as fout:
    r = csv.reader(fin)
    w = csv.writer(fout)
    for row in r:
        if not row:
            continue
        bits = [("'" + str_to_bits_q5_11(x)) for x in row if x.strip()]
        w.writerow(bits)

print("✅ 저장 완료:", out)


In [ ]:
import csv

def extract_16b_core(x: str):
    """셀 문자열에서 16비트(0/1)만 추출. 유효하지 않으면 None."""
    if x is None:
        return None
    s = x.strip()
    if not s:
        return None
    if s.startswith("'"):
        s = s[1:].strip()
    core = s
    if len(core) == 16 and set(core) <= {"0", "1"}:
        return core
    return None

def group_row_to_32bit_single_quote(cells):
    """한 행의 16비트 토큰들을 2개씩 묶어 '32b(='16b + 16b) 형태로 반환."""
    cores = []
    for c in cells:
        core = extract_16b_core(c)
        if core is not None:
            cores.append(core)

    out = []
    i = 0
    while i < len(cores):
        if i + 1 < len(cores):
            out.append("'" + cores[i] + cores[i+1])  # "'xxxxxxxxxxxxxxxxyyyyyyyyyyyyyyyy"
            i += 2
        else:
            out.append("'" + cores[i])                # 홀수 개면 마지막은 "'xxxxxxxxxxxxxxxx"
            i += 1
    return out

def process_file(inp_path, out_path):
    with open(inp_path, "r", newline="") as fin, open(out_path, "w", newline="") as fout:
        r = csv.reader(fin)
        w = csv.writer(fout)
        for row in r:
            if not row:
                w.writerow([])
                continue
            w.writerow(group_row_to_32bit_single_quote(row))
    print(f"✅ 저장 완료: {out_path}")

# 실행: layer0 / layer1 둘 다 처리
process_file("layer0_base_scaler_spline8_bits16.csv",
             "layer0_base_scaler_spline8_bits32_grouped.csv")
process_file("layer1_base_scaler_spline8_bits16.csv",
             "layer1_base_scaler_spline8_bits32_grouped.csv")


In [ ]:
import torch
import numpy as np

# 1️⃣ 파일 경로
file_path = "model/func.pth"

# 2️⃣ state_dict 로드
state_dict = torch.load(file_path, map_location="cpu")

# 3️⃣ 파라미터 정보 출력
print("===== 저장된 파라미터 목록 =====")
for k in state_dict.keys():
    t = state_dict[k]
    print(f"{k}: shape={t.shape}, dtype={t.dtype}")

# 4️⃣ 모든 파라미터를 하나의 리스트에 합치기 + CSV 파일로 저장
all_weights = []

for k in state_dict.keys():
    arr = state_dict[k].cpu().numpy().flatten()
    all_weights.extend(arr)

# 전체 파라미터를 하나의 큰 CSV 파일에 저장
flat_csv_file = "all_weights_flat.csv"
np.savetxt(flat_csv_file, np.array(all_weights), delimiter=",")
print(f"\n✅ 전체 파라미터 CSV '{flat_csv_file}' 저장 완료!")

# 5️⃣ 각 파라미터를 별도의 CSV 파일로 저장
print("\n===== 개별 CSV 파일 저장 =====")
for k in state_dict.keys():
    arr = state_dict[k].cpu().numpy()
    filename = k.replace(".", "_") + ".csv"
    # 2D 이상은 flatten하지 않고 원래 shape 유지
    np.savetxt(filename, arr.reshape(-1, arr.shape[-1]) if arr.ndim > 1 else arr, delimiter=",")
    print(f"{filename} 저장 완료")


In [ ]:
def float_to_custom_bits(value: float) -> str:
    """
    주어진 실수 value를 부호1비트, 정수부11비트, 소수부20비트(총 32비트)의 고정소수점 표현으로 변환한다.
    음수는 전체 32비트를 2의 보수로 표현한다.
    """
    scale = 2 ** 20  # 소수부 20비트에 해당하는 스케일
    fixed_val = int(round(value * scale))
    twos_comp = fixed_val & 0xFFFFFFFF  # 32비트 2의 보수 표현
    return format(twos_comp, '032b')

float_to_custom_bits(-0.50041)

In [ ]:
def float_to_custom_bits(value: float) -> str:
    """
    주어진 실수 value를 부호1비트, 정수부11비트, 소수부20비트(총 32비트)의 고정소수점 표현으로 변환한다.
    음수는 전체 32비트를 2의 보수로 표현한다.
    """
    scale = 2 ** 20  # 소수부 20비트에 해당하는 스케일
    fixed_val = int(round(value * scale))
    twos_comp = fixed_val & 0xFFFFFFFF  # 32비트 2의 보수 표현
    return format(twos_comp, '032b')

def custom_bits_to_float(bit_str: str) -> float:
    """
    32비트 2진수 문자열(bit_str)을 고정소수점 표현으로 해석하여 원래의 실수로 복원한다.
    """
    if len(bit_str) != 32:
        raise ValueError("입력된 문자열은 반드시 32비트여야 합니다.")
    value = int(bit_str, 2)
    # 최상위 비트가 1이면 음수 (2의 보수 보정)
    if bit_str[0] == '1':
        value -= 2 ** 32
    scale = 2 ** 20
    return value / scale

print(custom_bits_to_float('11111111111111011011010101000110')) #fffdb546
print(custom_bits_to_float('00000000000001011011110011000011')) #5bcc3
print(custom_bits_to_float('11111111111111011000100001110110')) #fffd8876
print(custom_bits_to_float('00000000000000000001100110001010')) #198a

print(custom_bits_to_float('00000000000000100110110100001101')) #26d0d
print(custom_bits_to_float('00000000000010011111000010000011')) #9f083
print(custom_bits_to_float('11111111111110110111110101110110')) #fffb7d76
print(custom_bits_to_float('00000000000000000010010100110011')) #2533

print(custom_bits_to_float('00000000001001001111011010100101'))


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# 표준 SiLU 함수 (벡터/스칼라 모두 지원)
def silu(x):
    return x / (1 + np.exp(-x))

def float_to_custom_bits(value: float) -> str:
    """
    주어진 실수 value를 부호1비트, 정수부11비트, 소수부20비트(총 32비트)의 고정소수점 표현으로 변환한다.
    음수는 전체 32비트를 2의 보수로 표현한다.
    """
    scale = 2 ** 20  # 소수부 20비트에 해당하는 스케일
    fixed_val = int(round(value * scale))
    twos_comp = fixed_val & 0xFFFFFFFF  # 32비트 2의 보수 표현
    return format(twos_comp, '032b')

def custom_bits_to_float(bit_str: str) -> float:
    """
    32비트 2진수 문자열(bit_str)을 고정소수점 표현으로 해석하여 원래의 실수로 복원한다.
    """
    if len(bit_str) != 32:
        raise ValueError("입력된 문자열은 반드시 32비트여야 합니다.")
    value = int(bit_str, 2)
    # 최상위 비트가 1이면 음수 (2의 보수 보정)
    if bit_str[0] == '1':
        value -= 2 ** 32
    scale = 2 ** 20
    return value / scale

def bits_to_fixed_int(bit_str: str) -> int:
    """
    32비트 2진 문자열(bit_str)을 2의 보수 해석에 따른 정수로 변환한다.
    """
    value = int(bit_str, 2)
    if bit_str[0] == '1':
        value -= (1 << 32)
    return value

def fixed_int_to_bits(fixed_val: int) -> str:
    """
    정수 fixed_val을 32비트 2의 보수 형식의 2진 문자열로 변환한다.
    """
    return format(fixed_val & 0xFFFFFFFF, '032b')

def add_bits(bit1: str, bit2: str) -> str:
    """
    32비트 고정소수점 표현(bit1, bit2)을 덧셈한 결과를 32비트 2의 보수 문자열로 반환한다.
    """
    a = bits_to_fixed_int(bit1)
    b = bits_to_fixed_int(bit2)
    result = a + b
    return fixed_int_to_bits(result)

def sub_bits(bit1: str, bit2: str) -> str:
    """
    32비트 고정소수점 표현(bit1, bit2)에서 빼기 연산(bit1 - bit2)을 수행한 결과를 반환한다.
    """
    a = bits_to_fixed_int(bit1)
    b = bits_to_fixed_int(bit2)
    result = a - b
    return fixed_int_to_bits(result)

def mul_bits(bit1: str, bit2: str) -> str:
    """
    32비트 고정소수점 표현(bit1, bit2)을 곱한다.

    두 숫자 모두 내부적으로 스케일 2^20를 사용하므로,
    곱한 후 결과를 다시 32비트 고정소수점 형식으로 맞추기 위해 오른쪽으로 20비트 시프트한다.
    """
    a = bits_to_fixed_int(bit1)
    b = bits_to_fixed_int(bit2)
    product = (a * b) >> 20
    return fixed_int_to_bits(product)

def square_bits(bit: str) -> str:
    """
    주어진 32비트 고정소수점 표현을 제곱하여 32비트 2의 보수 형식으로 반환한다.
    """
    return mul_bits(bit, bit)

def cube_bits(bit: str) -> str:
    """
    주어진 32비트 고정소수점 표현을 세제곱하여 32비트 2의 보수 형식으로 반환한다.
    """
    return mul_bits(bit, square_bits(bit))

def extract_int_bits_from_custom(bit_str: str) -> str:
    """
    입력된 32비트 고정소수점 표현(bit_str)에서 정수부(바닥(floor) 방식)를 남긴다.
    음수일 때 -3.25 -> 정수부 = -4
    """
    # 전체 실수값 복원
    value = custom_bits_to_float(bit_str)
    # floor 방식으로 정수부 추출
    int_value = math.floor(value)
    return float_to_custom_bits(int_value)


def extract_frac_bits_from_custom(bit_str: str) -> str:
    """
    입력된 32비트 고정소수점 표현(bit_str)에서 소수부(0~1)만 남긴다.
    음수일 때 -3.25 -> frac = 0.75
    """
    # 전체 실수값 복원
    value = custom_bits_to_float(bit_str)
    # floor 방식으로 정수부
    int_value = math.floor(value)
    # fractional part 계산 (양수 분수)
    frac_value = value - int_value
    return float_to_custom_bits(frac_value)

def get_initial_guess(denom, x):
    d = custom_bits_to_float(denom)
    if x < 0:
        # 음수 x에 대한 초기 추정값 테이블
        if d > 130:  # 246
            return const_0_0065   # x ≈ -0.5
        else:
            return const_0_01  # 극단적으로 작을 경우
    else:
        # 양수 x에 대한 기존 추정값 테이블
        if d > 200:
            return const_0_004    # x ≈ 1.5
        elif d > 130:
          return const_0_0065
        else:
            return const_0_00653  # x ≥ 5.5



def newton_division(num, denom, x, iterations):
    guess = get_initial_guess(denom, x)
    #print("initial",x, custom_bits_to_float(denom), custom_bits_to_float(guess) )
    for _ in range(iterations):
        guess = mul_bits(guess, sub_bits(const2, mul_bits(denom, guess)))
        #print(x, custom_bits_to_float(denom),custom_bits_to_float(guess))
    return mul_bits(num, guess)

def custom_silu(y, iterations):
    x = float_to_custom_bits(y)
    integer = extract_int_bits_from_custom(x)
    float_integer = custom_bits_to_float(integer)
    minus_float_integer = -float_integer
    frac = extract_frac_bits_from_custom(x)
    if y > 0:
        first_numerator = const120
        secon_numerator = mul_bits(const60, frac)
        third_numerator = mul_bits(const12, mul_bits(frac, frac))
        fourth_numerator = cube_bits(frac)
        numerator = mul_bits(x, add_bits(add_bits(add_bits(first_numerator, secon_numerator), third_numerator), fourth_numerator))
        #numerator = mul_bits(x, add_bits(add_bits(first_numerator, secon_numerator), third_numerator))



        lut1 = add_bits(const1, float_to_custom_bits(math.exp(minus_float_integer)))
        lut2 = sub_bits(const1,float_to_custom_bits(math.exp(minus_float_integer)))
        first_denominator = mul_bits(const12, mul_bits(lut1, add_bits(const10, square_bits(frac))))
        second_denominator = mul_bits(lut2, mul_bits(frac, add_bits(const60, square_bits(frac))))
        denominator = add_bits(first_denominator, second_denominator)

        result = newton_division(numerator, denominator, y, iterations)
        return custom_bits_to_float(result)
    elif y < 0:
        first_numerator = const120
        secon_numerator = mul_bits(const60, frac)
        third_numerator = mul_bits(const12, mul_bits(frac, frac))
        fourth_numerator = cube_bits(frac)
        numerator = mul_bits(float_to_custom_bits(math.exp(float_integer)), mul_bits(x, add_bits(add_bits(add_bits(first_numerator, secon_numerator), third_numerator), fourth_numerator)))
        #numerator = mul_bits(float_to_custom_bits(math.exp(float_integer)), mul_bits(x, add_bits(add_bits(first_numerator, secon_numerator), third_numerator)))

        lut1 = add_bits(const1, float_to_custom_bits(math.exp(float_integer)))
        lut2 = sub_bits(float_to_custom_bits(math.exp(float_integer)),const1)
        first_denominator = mul_bits(const12, mul_bits(lut1, add_bits(const10, square_bits(frac))))
        second_denominator = mul_bits(lut2, mul_bits(frac, add_bits(const60, square_bits(frac))))
        denominator = add_bits(first_denominator, second_denominator)

        #print(custom_bits_to_float(denominator))

        result = newton_division(numerator, denominator, y, iterations)
        return custom_bits_to_float(result)

# 상수 정의
const2 = float_to_custom_bits(2)
const120 = float_to_custom_bits(120)
const12 = float_to_custom_bits(12)
const60 = float_to_custom_bits(60)
const1 = float_to_custom_bits(1)
const10 = float_to_custom_bits(10)
const_0_01 = float_to_custom_bits(0.01)

# 정수 상수들
const180 = float_to_custom_bits(180)
const149 = float_to_custom_bits(149)
const113 = float_to_custom_bits(113)
const100 = float_to_custom_bits(100)
const95 = float_to_custom_bits(95)

# 소수 상수들 (첫번째 그룹)
const_0_004 = float_to_custom_bits(0.004)
const_0_0067 = float_to_custom_bits(0.0067)
const_0_0088 = float_to_custom_bits(0.0088)
const_0_00995 = float_to_custom_bits(0.00995)
const_0_01045 = float_to_custom_bits(0.01045)
const_0_00108 = float_to_custom_bits(0.00108)

# 정수 상수들 (두번째 그룹)
const200 = float_to_custom_bits(200)
const160 = float_to_custom_bits(160)
const145 = float_to_custom_bits(145)
const130 = float_to_custom_bits(130)

# 소수 상수들 (두번째 그룹)
const_0_004_2 = float_to_custom_bits(0.004)
const_0_0060 = float_to_custom_bits(0.006)
const_0_0063 = float_to_custom_bits(0.0063)
const_0_0065 = float_to_custom_bits(0.0065)
const_0_00653 = float_to_custom_bits(0.00653)
const_0_065 = float_to_custom_bits(0.065)

const_e0 = float_to_custom_bits(math.exp(0))
const_e_neg1 = float_to_custom_bits(math.exp(-1))
const_e_neg2 = float_to_custom_bits(math.exp(-2))
const_e_neg3 = float_to_custom_bits(math.exp(-3))
const_e_neg4 = float_to_custom_bits(math.exp(-4))
const_e_neg5 = float_to_custom_bits(math.exp(-5))
const_e_neg6 = float_to_custom_bits(math.exp(-6))
const_e_neg7 = float_to_custom_bits(math.exp(-7))
const_e_neg8 = float_to_custom_bits(math.exp(-8))
const_e_neg9 = float_to_custom_bits(math.exp(-9))
const_e_neg10 = float_to_custom_bits(math.exp(-10))

####################################################################
# 테스트: x 범위를 [0.1,10]에서 표준 SiLU와 custom_silu 결과를 비교
####################################################################
x_vals = np.linspace(-10, 10, 1000)
y_true = silu(x_vals)

# custom_silu를 배열의 각 요소에 적용
y_custom = np.array([custom_silu(x, iterations=2) for x in x_vals])

# 오차 계산
abs_error = np.abs(y_true - y_custom)
epsilon = 1e-8
percent_error = abs_error / (np.abs(y_true) + epsilon) * 100

print("절대 오차: 최대 =", np.max(abs_error), ", 최소 =", np.min(abs_error))
print("퍼센트 오차: 최대 =", np.max(percent_error), ", 최소 =", np.min(percent_error))

# 그래프 1: SiLU 함수 비교
plt.figure(figsize=(15,5))
plt.subplot(1,3,1)
plt.plot(x_vals, y_true, label="True SiLU", linewidth=2)
plt.plot(x_vals, y_custom, label="Custom SiLU (Newton)", linestyle="--", linewidth=2)
plt.title("SiLU 함수 비교")
plt.xlabel("x")
plt.ylabel("SiLU(x)")
plt.legend()
plt.grid(True)

# 그래프 2: 절대 오차
plt.subplot(1,3,2)
plt.plot(x_vals, abs_error, color="red", linewidth=2)
plt.title("abs error")
plt.xlabel("x")
plt.ylabel("abs error")
plt.grid(True)

# 그래프 3: 퍼센트 오차
plt.subplot(1,3,3)
plt.plot(x_vals, percent_error, color="green", linewidth=2)
plt.title("per error")
plt.xlabel("x")
plt.ylabel("per error (%)")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# ===== 고정소수점 설정 =====
FRAC_BITS = 11                 # 소수부 비트 수
SCALE = 1 << FRAC_BITS         # = 2^11
TOTAL_BITS = 32
MASK32 = (1 << TOTAL_BITS) - 1

# 표준 SiLU 함수 (벡터/스칼라 모두 지원)
def silu(x):
    return x / (1 + np.exp(-x))

def float_to_custom_bits(value: float) -> str:
    """
    주어진 실수 value를 부호1비트, 정수부20비트, 소수부11비트(총 32비트)의
    2의 보수 고정소수점 표현으로 변환한다.
    음수는 전체 32비트를 2의 보수로 표현한다.
    """
    fixed_val = int(round(value * SCALE))
    twos_comp = fixed_val & MASK32  # 32비트 2의 보수 표현으로 맞춤
    return format(twos_comp, '032b')

def custom_bits_to_float(bit_str: str) -> float:
    """
    32비트 2진수 문자열(bit_str)을 고정소수점(소수부 11비트)으로 해석하여 실수로 복원한다.
    """
    if len(bit_str) != 32:
        raise ValueError("입력된 문자열은 반드시 32비트여야 합니다.")
    value = int(bit_str, 2)
    # 최상위 비트가 1이면 음수 (2의 보수 보정)
    if bit_str[0] == '1':
        value -= 1 << TOTAL_BITS
    return value / SCALE

def bits_to_fixed_int(bit_str: str) -> int:
    """
    32비트 2진 문자열(bit_str)을 2의 보수 해석에 따른 정수로 변환한다.
    (내부 고정소수점 스케일이 적용된 정수 값)
    """
    value = int(bit_str, 2)
    if bit_str[0] == '1':
        value -= (1 << TOTAL_BITS)
    return value

def fixed_int_to_bits(fixed_val: int) -> str:
    """
    정수 fixed_val(스케일 적용된 값)을 32비트 2의 보수 형식의 2진 문자열로 변환한다.
    """
    return format(fixed_val & MASK32, '032b')

def add_bits(bit1: str, bit2: str) -> str:
    """
    32비트 고정소수점 표현(bit1, bit2)을 덧셈한 결과를 32비트 2의 보수 문자열로 반환한다.
    """
    a = bits_to_fixed_int(bit1)
    b = bits_to_fixed_int(bit2)
    result = a + b
    return fixed_int_to_bits(result)

def sub_bits(bit1: str, bit2: str) -> str:
    """
    32비트 고정소수점 표현(bit1, bit2)에서 빼기 연산(bit1 - bit2)을 수행한 결과를 반환한다.
    """
    a = bits_to_fixed_int(bit1)
    b = bits_to_fixed_int(bit2)
    result = a - b
    return fixed_int_to_bits(result)

def mul_bits(bit1: str, bit2: str) -> str:
    """
    32비트 고정소수점 표현(bit1, bit2)을 곱한다.
    두 숫자 모두 내부적으로 스케일 2^FRAC_BITS(=2^11)를 사용하므로,
    곱한 후 결과를 다시 32비트 고정소수점 형식으로 맞추기 위해 오른쪽으로 FRAC_BITS만큼 시프트한다.
    """
    a = bits_to_fixed_int(bit1)
    b = bits_to_fixed_int(bit2)
    product = (a * b) >> FRAC_BITS
    return fixed_int_to_bits(product)

def square_bits(bit: str) -> str:
    """
    주어진 32비트 고정소수점 표현을 제곱하여 32비트 2의 보수 형식으로 반환한다.
    """
    return mul_bits(bit, bit)

def cube_bits(bit: str) -> str:
    """
    주어진 32비트 고정소수점 표현을 세제곱하여 32비트 2의 보수 형식으로 반환한다.
    """
    return mul_bits(bit, square_bits(bit))

def extract_int_bits_from_custom(bit_str: str) -> str:
    """
    입력된 32비트 고정소수점 표현(bit_str)에서 정수부(바닥(floor) 방식)를 남긴다.
    예) 음수일 때 -3.25 -> 정수부 = -4
    """
    value = custom_bits_to_float(bit_str)
    int_value = math.floor(value)
    return float_to_custom_bits(int_value)

def extract_frac_bits_from_custom(bit_str: str) -> str:
    """
    입력된 32비트 고정소수점 표현(bit_str)에서 소수부(0~1)만 남긴다.
    예) 음수일 때 -3.25 -> frac = 0.75
    """
    value = custom_bits_to_float(bit_str)
    int_value = math.floor(value)
    frac_value = value - int_value
    return float_to_custom_bits(frac_value)

"""
def get_initial_guess(denom, x):
    d = custom_bits_to_float(denom)
    if x < 0:
        # 음수 x에 대한 초기 추정값 테이블
        if d > 130:  # 246
            return const_0_0065   # x ≈ -0.5
        else:
            return const_0_01  # 극단적으로 작을 경우
    else:
        # 양수 x에 대한 기존 추정값 테이블
        if d > 200:
            return const_0_004    # x ≈ 1.5
        elif d > 130:
          return const_0_0065
        else:
            return const_0_00653  # x ≥ 5.5
"""
def get_initial_guess(denom, x):
    d = custom_bits_to_float(denom)
    if d > 200:
        return const_0_004    # x ≈ 1.5
    elif d > 130:
        return const_0_0065
    else:
        return const_0_01  # x ≥ 5.5

def newton_division(num, denom, x, iterations):
    guess = get_initial_guess(denom, x)
    #print("initial",x, custom_bits_to_float(denom), custom_bits_to_float(guess) )
    for _ in range(iterations):
        guess = mul_bits(guess, sub_bits(const2, mul_bits(denom, guess)))
        #print(x, custom_bits_to_float(denom),custom_bits_to_float(guess))
    return mul_bits(num, guess)

def custom_silu(y, iterations):
    x = float_to_custom_bits(y)
    integer = extract_int_bits_from_custom(x)
    float_integer = custom_bits_to_float(integer)
    minus_float_integer = -float_integer
    frac = extract_frac_bits_from_custom(x)
    if y > 0:
        first_numerator = const120
        secon_numerator = mul_bits(const60, frac)
        third_numerator = mul_bits(const12, mul_bits(frac, frac))
        fourth_numerator = cube_bits(frac)
        numerator = mul_bits(x, add_bits(add_bits(add_bits(first_numerator, secon_numerator), third_numerator), fourth_numerator))
        #numerator = mul_bits(x, add_bits(add_bits(first_numerator, secon_numerator), third_numerator))



        lut1 = add_bits(const1, float_to_custom_bits(math.exp(minus_float_integer)))
        lut2 = sub_bits(const1,float_to_custom_bits(math.exp(minus_float_integer)))
        first_denominator = mul_bits(const12, mul_bits(lut1, add_bits(const10, square_bits(frac))))
        second_denominator = mul_bits(lut2, mul_bits(frac, add_bits(const60, square_bits(frac))))
        denominator = add_bits(first_denominator, second_denominator)

        result = newton_division(numerator, denominator, y, iterations)
        return custom_bits_to_float(result)
    elif y < 0:
        first_numerator = const120
        secon_numerator = mul_bits(const60, frac)
        third_numerator = mul_bits(const12, mul_bits(frac, frac))
        fourth_numerator = cube_bits(frac)
        numerator = mul_bits(float_to_custom_bits(math.exp(float_integer)), mul_bits(x, add_bits(add_bits(add_bits(first_numerator, secon_numerator), third_numerator), fourth_numerator)))
        #numerator = mul_bits(float_to_custom_bits(math.exp(float_integer)), mul_bits(x, add_bits(add_bits(first_numerator, secon_numerator), third_numerator)))

        lut1 = add_bits(const1, float_to_custom_bits(math.exp(float_integer)))
        lut2 = sub_bits(float_to_custom_bits(math.exp(float_integer)),const1)
        first_denominator = mul_bits(const12, mul_bits(lut1, add_bits(const10, square_bits(frac))))
        second_denominator = mul_bits(lut2, mul_bits(frac, add_bits(const60, square_bits(frac))))
        denominator = add_bits(first_denominator, second_denominator)

        #print(custom_bits_to_float(denominator))

        result = newton_division(numerator, denominator, y, iterations)
        return custom_bits_to_float(result)

# 상수 정의
const2 = float_to_custom_bits(2)
const120 = float_to_custom_bits(120)
const12 = float_to_custom_bits(12)
const60 = float_to_custom_bits(60)
const1 = float_to_custom_bits(1)
const10 = float_to_custom_bits(10)
const_0_01 = float_to_custom_bits(0.01)

# 정수 상수들
const180 = float_to_custom_bits(180)
const149 = float_to_custom_bits(149)
const113 = float_to_custom_bits(113)
const100 = float_to_custom_bits(100)
const95 = float_to_custom_bits(95)

# 소수 상수들 (첫번째 그룹)
const_0_004 = float_to_custom_bits(0.004)
const_0_0067 = float_to_custom_bits(0.0067)
const_0_0088 = float_to_custom_bits(0.0088)
const_0_00995 = float_to_custom_bits(0.00995)
const_0_01045 = float_to_custom_bits(0.01045)
const_0_00108 = float_to_custom_bits(0.00108)

# 정수 상수들 (두번째 그룹)
const200 = float_to_custom_bits(200)
const160 = float_to_custom_bits(160)
const145 = float_to_custom_bits(145)
const130 = float_to_custom_bits(130)

# 소수 상수들 (두번째 그룹)
const_0_004_2 = float_to_custom_bits(0.004)
const_0_0060 = float_to_custom_bits(0.006)
const_0_0063 = float_to_custom_bits(0.0063)
const_0_0065 = float_to_custom_bits(0.0065)
const_0_00653 = float_to_custom_bits(0.00653)
const_0_065 = float_to_custom_bits(0.065)

const_e0 = float_to_custom_bits(math.exp(0))
const_e_neg1 = float_to_custom_bits(math.exp(-1))
const_e_neg2 = float_to_custom_bits(math.exp(-2))
const_e_neg3 = float_to_custom_bits(math.exp(-3))
const_e_neg4 = float_to_custom_bits(math.exp(-4))
const_e_neg5 = float_to_custom_bits(math.exp(-5))
const_e_neg6 = float_to_custom_bits(math.exp(-6))
const_e_neg7 = float_to_custom_bits(math.exp(-7))
const_e_neg8 = float_to_custom_bits(math.exp(-8))
const_e_neg9 = float_to_custom_bits(math.exp(-9))
const_e_neg10 = float_to_custom_bits(math.exp(-10))

####################################################################
# 테스트: x 범위를 [0.1,10]에서 표준 SiLU와 custom_silu 결과를 비교
####################################################################
x_vals = np.linspace(-4, 1, 1000)
y_true = silu(x_vals)

# custom_silu를 배열의 각 요소에 적용
y_custom = np.array([custom_silu(x, iterations=2) for x in x_vals])

# 오차 계산
abs_error = np.abs(y_true - y_custom)
epsilon = 1e-8
percent_error = abs_error / (np.abs(y_true) + epsilon) * 100

print("절대 오차: 최대 =", np.max(abs_error), ", 최소 =", np.min(abs_error))
print("퍼센트 오차: 최대 =", np.max(percent_error), ", 최소 =", np.min(percent_error))

# 그래프 1: SiLU 함수 비교
plt.figure(figsize=(15,5))
plt.subplot(1,3,1)
plt.plot(x_vals, y_true, label="True SiLU", linewidth=2)
plt.plot(x_vals, y_custom, label="Custom SiLU (Newton)", linestyle="--", linewidth=2)
plt.title("SiLU 함수 비교")
plt.xlabel("x")
plt.ylabel("SiLU(x)")
plt.legend()
plt.grid(True)

# 그래프 2: 절대 오차
plt.subplot(1,3,2)
plt.plot(x_vals, abs_error, color="red", linewidth=2)
plt.title("abs error")
plt.xlabel("x")
plt.ylabel("abs error")
plt.grid(True)

# 그래프 3: 퍼센트 오차
plt.subplot(1,3,3)
plt.plot(x_vals, percent_error, color="green", linewidth=2)
plt.title("per error")
plt.xlabel("x")
plt.ylabel("per error (%)")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
FRAC_BITS = 11
SCALE = 1 << FRAC_BITS      # 2048
INT16_MIN = -(1 << 15)      # -32768
INT16_MAX =  (1 << 15) - 1  #  32767
import math

def float_bit(x: float) -> str:
    """
    float -> Q5.11 (출력: 16비트 2진 문자열)
    """
    raw = int(round(x * SCALE))
    # int16 범위로 클리핑
    if raw < INT16_MIN: raw = INT16_MIN
    elif raw > INT16_MAX: raw = INT16_MAX
    bits = raw & 0xFFFF   # 16비트 패턴
    return format(bits, '016b')  # 16자리 2진수 문자열

def bit_float(bits_str: str) -> float:
    """
    Q5.11 (입력: 16비트 2진 문자열) -> float
    """
    v = int(bits_str, 2)   # 0~65535
    if v & 0x8000:         # 부호비트 확인
        v -= 1 << 16
    return v / SCALE

#print(float_bit(-1))
#print(float_bit(4.8))

#print(bit_float('0010010001010101'))

print(float_bit(-1.2428418397903442))

print(float_bit(0.3166329))
print(float_bit(0.2205670))
print(float_bit(0.4183821))
print(float_bit(-0.2696311))
print(float_bit(0.2374790))
print(float_bit(-0.4523106))
print(float_bit(-0.1779861))
print(float_bit(0.1841769))

print(float_bit(0.5))


print(bit_float('1111110100000110'))
print(bit_float('1111111001000110'))
print(bit_float('1111101101001100'))


